# CEAL Inference — GazeNet Cross-Dataset Evaluation

**Goal:** Run inference on the CEAL dataset (5,880 images, 56 subjects, 5 classes) using our best GazeCapture-trained GazeNet models. No training, no finetuning.

**Baseline to beat:** iTracker achieved 39% accuracy on this same dataset.

**Supports three model variants:**
- `GazeNet` (v16A) — 3 image streams, no geo features
- `GazeNetV17` (m2) — 3 image streams + 7 geo features
- `GazeNetV17b` (m3) — same as m2, smaller FC head

**Inputs from external drive:**
- CEAL crops (224×224): `/Volumes/Crucial X10/210/data/ceal_itracker_artifacts/`
- CEAL geo features: `/Volumes/Crucial X10/210/ceal_geo_features_v1.parquet`
- Manifest CSV: `/Volumes/Crucial X10/210/data/ceal_itracker_artifacts/manifest.csv`
- Model checkpoints: `/content/drive/MyDrive/210/` (or local path)

# Setup

In [ ]:
import re
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Paths

Update these to match your local setup.

In [ ]:
# ============================================================
# ALL PATHS — update these to match your machine
# ============================================================

MANIFEST_CSV     = "/Volumes/Crucial X10/210/data/ceal_itracker_artifacts/manifest.csv"
GEO_PARQUET      = "/Volumes/Crucial X10/210/ceal_geo_features_v1.parquet"

# Model checkpoints — pick one or run all three
CHECKPOINTS = {
    'v16A': '/content/drive/MyDrive/checkpoints/best_gazenet_model_v16A.pth',
    'm2':   '/content/drive/MyDrive/210/m2.pth',
    'm3':   '/content/drive/MyDrive/210/m3.pth',
}

# Which model to evaluate (change this to run a different one)
ACTIVE_MODEL = 'm2'   # 'v16A', 'm2', or 'm3'

print(f"Active model: {ACTIVE_MODEL}")
print(f"Checkpoint: {CHECKPOINTS[ACTIVE_MODEL]}")

# Model Definitions

All three GazeNet variants, copied from training notebooks for checkpoint compatibility.

In [ ]:
# ============================================================
# GazeNet (v16A) — baseline, no geo features
# forward(left_eye, right_eye, face) -> (batch, 5)
# ============================================================

class GazeNet(nn.Module):
    def __init__(self, num_classes=5):
        super(GazeNet, self).__init__()

        self.eye_cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.face_cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=7, stride=2, padding=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.fc = nn.Sequential(
            nn.Linear(128*6*6*2 + 256*3*3, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, left_eye, right_eye, face):
        left_flat = self.eye_cnn(left_eye).view(left_eye.size(0), -1)
        right_flat = self.eye_cnn(right_eye).view(right_eye.size(0), -1)
        face_flat = self.face_cnn(face).view(face.size(0), -1)
        combined = torch.cat([left_flat, right_flat, face_flat], dim=1)
        return self.fc(combined)


# ============================================================
# GazeNetV17 (m2) — with geo feature branch
# forward(left_eye, right_eye, face, geo_features) -> (batch, 5)
# ============================================================

class GazeNetV17(nn.Module):
    def __init__(self, num_classes=5, geo_feat_dim=7):
        super(GazeNetV17, self).__init__()

        self.eye_cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.face_cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=7, stride=2, padding=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.geo_mlp = nn.Sequential(
            nn.Linear(geo_feat_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 64),
            nn.ReLU(),
        )

        self.fc = nn.Sequential(
            nn.Linear(4608 * 2 + 2304 + 64, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, left_eye, right_eye, face, geo_features):
        left_feat  = self.eye_cnn(left_eye).view(left_eye.size(0), -1)
        right_feat = self.eye_cnn(right_eye).view(right_eye.size(0), -1)
        face_feat  = self.face_cnn(face).view(face.size(0), -1)
        geo_feat   = self.geo_mlp(geo_features)
        combined = torch.cat([left_feat, right_feat, face_feat, geo_feat], dim=1)
        return self.fc(combined)


# ============================================================
# GazeNetV17b (m3) — smaller FC, higher dropout
# forward(left_eye, right_eye, face, geo_features) -> (batch, 5)
# ============================================================

class GazeNetV17b(nn.Module):
    def __init__(self, num_classes=5, geo_feat_dim=7):
        super(GazeNetV17b, self).__init__()

        self.eye_cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.face_cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=7, stride=2, padding=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.geo_mlp = nn.Sequential(
            nn.Linear(geo_feat_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 64),
            nn.ReLU(),
        )

        self.fc = nn.Sequential(
            nn.Linear(4608 * 2 + 2304 + 64, 256),
            nn.ReLU(),
            nn.Dropout(0.7),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.7),
            nn.Linear(128, num_classes),
        )

    def forward(self, left_eye, right_eye, face, geo_features):
        left_feat  = self.eye_cnn(left_eye).view(left_eye.size(0), -1)
        right_feat = self.eye_cnn(right_eye).view(right_eye.size(0), -1)
        face_feat  = self.face_cnn(face).view(face.size(0), -1)
        geo_feat   = self.geo_mlp(geo_features)
        combined = torch.cat([left_feat, right_feat, face_feat, geo_feat], dim=1)
        return self.fc(combined)


# ============================================================
# Model registry — maps ACTIVE_MODEL string to class
# ============================================================
MODEL_CLASSES = {
    'v16A': GazeNet,
    'm2':   GazeNetV17,
    'm3':   GazeNetV17b,
}

# Models that accept geo_features as a 4th input
MODELS_WITH_GEO = {'m2', 'm3'}

print("Model definitions loaded")
for name, cls in MODEL_CLASSES.items():
    m = cls(num_classes=5)
    n_params = sum(p.numel() for p in m.parameters())
    geo_tag = " + geo" if name in MODELS_WITH_GEO else ""
    print(f"  {name}: {cls.__name__} — {n_params:,} params{geo_tag}")

# Load CEAL Data

In [ ]:
# ============================================================
# LOAD MANIFEST — filter to aug_id=0 (unaugmented originals)
# ============================================================

manifest = pd.read_csv(MANIFEST_CSV)
print(f"Full manifest: {len(manifest)} rows")
print(f"Status breakdown:\n{manifest['status'].value_counts()}")

# Keep only successful, unaugmented samples
df = manifest[(manifest['status'] == 'ok') & (manifest['aug_id'] == 0)].copy()
print(f"\nAfter filtering (status=ok, aug_id=0): {len(df)} samples")
print(f"Subjects: {df['subject'].nunique()}")

In [ ]:
# ============================================================
# CREATE LABELS from gaze degrees
#
# Same logic as ceal_data_preprocessing.ipynb:
#   - straight: both 0
#   - horizontal dominates: left (h<0) or right (h>0)
#   - vertical dominates: down (v<0) or up (v>0)
#   - tie: horizontal wins
#
# Label map matches GazeNet training:
#   Straight=0, Up=1, Down=2, Left=3, Right=4
# ============================================================

label_map = {'Straight': 0, 'Up': 1, 'Down': 2, 'Left': 3, 'Right': 4}
idx_to_label = {v: k for k, v in label_map.items()}

def degrees_to_label(gaze_v, gaze_h):
    """Convert CEAL gaze degrees to class label string."""
    if gaze_v == 0 and gaze_h == 0:
        return 'Straight'
    if abs(gaze_h) > abs(gaze_v):
        return 'Left' if gaze_h < 0 else 'Right'
    if abs(gaze_v) > abs(gaze_h):
        return 'Down' if gaze_v < 0 else 'Up'
    # Tie → horizontal wins
    return 'Left' if gaze_h < 0 else 'Right'

df['label_str'] = df.apply(
    lambda r: degrees_to_label(r['gaze_vertical_deg'], r['gaze_horizontal_deg']), axis=1
)
df['label_idx'] = df['label_str'].map(label_map)

print("Label distribution:")
print(df['label_str'].value_counts())
print(f"\nTotal labeled samples: {len(df)}")

In [ ]:
# ============================================================
# LOAD GEO FEATURES (for m2/m3 models)
# Join on orig_filename from manifest ↔ filename from parquet
# ============================================================

use_geo = ACTIVE_MODEL in MODELS_WITH_GEO

geo_cols = ['left_iris_h', 'right_iris_h', 'iris_h_agreement',
            'head_yaw', 'head_pitch', 'z_tilt', 'z_nose_rel']

GEO_DEFAULT = np.array([0.5, 0.5, 0.0, 0.0, 0.35, -0.1, -0.26], dtype=np.float32)

if use_geo:
    df_geo = pd.read_parquet(GEO_PARQUET)
    print(f"Loaded geo features: {len(df_geo)} rows")

    # Build lookup: filename -> 7-dim numpy array
    geo_lookup = {}
    for _, row in df_geo.iterrows():
        geo_lookup[row['filename']] = row[geo_cols].values.astype(np.float32)

    # Check join coverage
    matched = df['orig_filename'].isin(geo_lookup).sum()
    print(f"Geo features matched: {matched}/{len(df)} ({100*matched/len(df):.1f}%)")
    if matched < len(df):
        print(f"  {len(df) - matched} samples will use GEO_DEFAULT")
else:
    geo_lookup = {}
    print(f"Model '{ACTIVE_MODEL}' does not use geo features — skipping")

# Dataset & DataLoader

In [ ]:
# ============================================================
# TRANSFORMS — match GazeNet training preprocessing
#   Eyes:  224×224 → 48×48, normalize(0.5, 0.5)
#   Face:  224×224 → 112×112, normalize(0.5, 0.5)
# ============================================================

eye_transform = transforms.Compose([
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

face_transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

print("Transforms ready")

In [ ]:
# ============================================================
# CEAL DATASET CLASS
#
# Reads pre-cropped images from manifest paths,
# applies GazeNet-compatible transforms, and optionally
# returns geo features for m2/m3 models.
# ============================================================

class CEALDataset(Dataset):
    def __init__(self, dataframe, eye_transform, face_transform,
                 geo_lookup=None, geo_default=None):
        self.df = dataframe.reset_index(drop=True)
        self.eye_transform = eye_transform
        self.face_transform = face_transform
        self.geo_lookup = geo_lookup or {}
        self.geo_default = geo_default

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Load images
        face      = Image.open(row['face_path']).convert('RGB')
        left_eye  = Image.open(row['left_path']).convert('RGB')
        right_eye = Image.open(row['right_path']).convert('RGB')

        # Apply transforms
        face      = self.face_transform(face)
        left_eye  = self.eye_transform(left_eye)
        right_eye = self.eye_transform(right_eye)

        label = torch.tensor(row['label_idx'], dtype=torch.long)

        sample = {
            'left_eye': left_eye,
            'right_eye': right_eye,
            'face': face,
            'label': label,
        }

        # Geo features (only used by m2/m3)
        if self.geo_lookup:
            geo = self.geo_lookup.get(row['orig_filename'])
            if geo is None:
                geo = self.geo_default.copy()
            sample['geo_features'] = torch.tensor(geo, dtype=torch.float32)

        return sample


# ============================================================
# CREATE DATALOADER
# ============================================================

dataset = CEALDataset(
    dataframe=df,
    eye_transform=eye_transform,
    face_transform=face_transform,
    geo_lookup=geo_lookup if use_geo else None,
    geo_default=GEO_DEFAULT if use_geo else None,
)

loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"Dataset: {len(dataset)} samples")
print(f"Batches: {len(loader)}")

In [ ]:
# ============================================================
# SMOKE TEST — verify one batch looks correct
# ============================================================

batch = next(iter(loader))
print("Batch shapes:")
print(f"  left_eye:  {batch['left_eye'].shape}")
print(f"  right_eye: {batch['right_eye'].shape}")
print(f"  face:      {batch['face'].shape}")
if 'geo_features' in batch:
    print(f"  geo_features: {batch['geo_features'].shape}")
print(f"  label:     {batch['label'].shape}")
print(f"  label sample: {batch['label'][:8].tolist()}")

# Load Model & Run Inference

In [ ]:
# ============================================================
# LOAD MODEL CHECKPOINT
# ============================================================

ModelClass = MODEL_CLASSES[ACTIVE_MODEL]
model = ModelClass(num_classes=5).to(device)

checkpoint_path = CHECKPOINTS[ACTIVE_MODEL]
state_dict = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {ACTIVE_MODEL} ({ModelClass.__name__}) — {n_params:,} params")
print(f"Checkpoint: {checkpoint_path}")

In [ ]:
# ============================================================
# INFERENCE LOOP
# ============================================================

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for batch in loader:
        left_eye  = batch['left_eye'].to(device)
        right_eye = batch['right_eye'].to(device)
        face      = batch['face'].to(device)
        labels    = batch['label'].to(device)

        if ACTIVE_MODEL in MODELS_WITH_GEO:
            geo_features = batch['geo_features'].to(device)
            outputs = model(left_eye, right_eye, face, geo_features)
        else:
            outputs = model(left_eye, right_eye, face)

        probs = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

accuracy = 100 * np.mean(all_preds == all_labels)
print(f"\n{'='*50}")
print(f"CEAL Inference Complete — Model: {ACTIVE_MODEL}")
print(f"{'='*50}")
print(f"Samples evaluated: {len(all_preds)}")
print(f"Overall accuracy:  {accuracy:.2f}%")
print(f"iTracker baseline: 39.00%")
print(f"Delta vs baseline: {accuracy - 39.0:+.2f}%")

# Evaluation

In [ ]:
# ============================================================
# CONFUSION MATRIX
# ============================================================

label_names = ['Straight', 'Up', 'Down', 'Left', 'Right']
cm = confusion_matrix(all_labels, all_preds)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title(f'{ACTIVE_MODEL} on CEAL — Counts (acc={accuracy:.1f}%)')

# Normalized (per-class recall)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=axes[1])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title(f'{ACTIVE_MODEL} on CEAL — Recall per class')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print(f"\n{'='*50}")
print(f"Classification Report — {ACTIVE_MODEL} on CEAL")
print(f"{'='*50}")
print(classification_report(all_labels, all_preds, target_names=label_names))

In [ ]:
# ============================================================
# CONFIDENCE ANALYSIS
#
# How confident is the model on correct vs incorrect predictions?
# For assistive tech, low-confidence predictions should be
# suppressed rather than acted on.
# ============================================================

max_probs = all_probs.max(axis=1)
correct_mask = all_preds == all_labels

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confidence distribution: correct vs incorrect
axes[0].hist(max_probs[correct_mask], bins=30, alpha=0.6, label=f'Correct (n={correct_mask.sum()})', color='green')
axes[0].hist(max_probs[~correct_mask], bins=30, alpha=0.6, label=f'Incorrect (n={(~correct_mask).sum()})', color='red')
axes[0].set_xlabel('Max softmax probability')
axes[0].set_ylabel('Count')
axes[0].set_title('Confidence: Correct vs Incorrect')
axes[0].legend()

# Accuracy at different confidence thresholds
thresholds = np.arange(0.3, 1.0, 0.05)
accs_at_thresh = []
coverage_at_thresh = []
for t in thresholds:
    mask = max_probs >= t
    if mask.sum() > 0:
        accs_at_thresh.append(100 * np.mean(all_preds[mask] == all_labels[mask]))
        coverage_at_thresh.append(100 * mask.mean())
    else:
        accs_at_thresh.append(np.nan)
        coverage_at_thresh.append(0)

ax1 = axes[1]
ax2 = ax1.twinx()
ax1.plot(thresholds, accs_at_thresh, 'b-o', markersize=4, label='Accuracy')
ax2.plot(thresholds, coverage_at_thresh, 'r-s', markersize=4, label='Coverage')
ax1.set_xlabel('Confidence threshold')
ax1.set_ylabel('Accuracy (%)', color='blue')
ax2.set_ylabel('Coverage (%)', color='red')
ax1.set_title('Accuracy vs Coverage at Confidence Thresholds')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

print(f"\nMean confidence (correct):   {max_probs[correct_mask].mean():.3f}")
print(f"Mean confidence (incorrect): {max_probs[~correct_mask].mean():.3f}")

In [ ]:
# ============================================================
# PER-SUBJECT ACCURACY
#
# CEAL has 56 subjects — do some subjects fail completely?
# This tells us whether domain shift affects all subjects
# equally or if it's subject-dependent.
# ============================================================

df_results = df[['subject', 'label_str', 'orig_filename']].copy()
df_results['pred_idx'] = all_preds
df_results['pred_str'] = [idx_to_label[p] for p in all_preds]
df_results['correct'] = (all_preds == all_labels)
df_results['confidence'] = all_probs.max(axis=1)

subject_acc = df_results.groupby('subject')['correct'].mean().sort_values()

fig, ax = plt.subplots(figsize=(14, 5))
subject_acc.plot(kind='bar', ax=ax, color='steelblue', alpha=0.8)
ax.axhline(y=accuracy/100, color='red', linestyle='--', label=f'Overall: {accuracy:.1f}%')
ax.axhline(y=0.39, color='orange', linestyle='--', label='iTracker: 39%')
ax.set_ylabel('Accuracy')
ax.set_title(f'Per-Subject Accuracy — {ACTIVE_MODEL} on CEAL')
ax.legend()
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.show()

print(f"\nSubject accuracy stats:")
print(f"  Min:    {100*subject_acc.min():.1f}% (subject {subject_acc.idxmin()})")
print(f"  Max:    {100*subject_acc.max():.1f}% (subject {subject_acc.idxmax()})")
print(f"  Median: {100*subject_acc.median():.1f}%")
print(f"  Std:    {100*subject_acc.std():.1f}%")

In [ ]:
# ============================================================
# HORIZONTAL vs VERTICAL BREAKDOWN
#
# GazeCapture mobile data is mostly horizontal gaze variation.
# CEAL has independent head pose + gaze, so vertical classes
# (Up/Down) may transfer worse than horizontal (Left/Right).
# ============================================================

horizontal_classes = {'Left', 'Right'}
vertical_classes = {'Up', 'Down'}

h_mask = df_results['label_str'].isin(horizontal_classes)
v_mask = df_results['label_str'].isin(vertical_classes)
s_mask = df_results['label_str'] == 'Straight'

h_acc = 100 * df_results.loc[h_mask, 'correct'].mean()
v_acc = 100 * df_results.loc[v_mask, 'correct'].mean()
s_acc = 100 * df_results.loc[s_mask, 'correct'].mean()

print(f"Accuracy by gaze axis:")
print(f"  Horizontal (Left/Right): {h_acc:.1f}% (n={h_mask.sum()})")
print(f"  Vertical   (Up/Down):    {v_acc:.1f}% (n={v_mask.sum()})")
print(f"  Straight:                {s_acc:.1f}% (n={s_mask.sum()})")

# Run All Models (Optional)

Compare all available checkpoints side-by-side on the same CEAL data.

In [ ]:
# ============================================================
# COMPARE ALL MODELS — run this cell to evaluate every
# checkpoint that exists on disk.
#
# Rebuilds the dataset with geo features enabled (needed for
# m2/m3) and runs inference for each model.
# ============================================================

# Ensure geo features are loaded
if not geo_lookup:
    df_geo = pd.read_parquet(GEO_PARQUET)
    for _, row in df_geo.iterrows():
        geo_lookup[row['filename']] = row[geo_cols].values.astype(np.float32)

# Dataset with geo features for all models
dataset_all = CEALDataset(
    dataframe=df,
    eye_transform=eye_transform,
    face_transform=face_transform,
    geo_lookup=geo_lookup,
    geo_default=GEO_DEFAULT,
)
loader_all = DataLoader(dataset_all, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

results_summary = []

for model_name, ckpt_path in CHECKPOINTS.items():
    if not os.path.exists(ckpt_path):
        print(f"  Skipping {model_name} — checkpoint not found: {ckpt_path}")
        continue

    print(f"\nEvaluating {model_name}...")
    ModelCls = MODEL_CLASSES[model_name]
    m = ModelCls(num_classes=5).to(device)
    m.load_state_dict(torch.load(ckpt_path, map_location=device))
    m.eval()

    preds = []
    labels = []

    with torch.no_grad():
        for batch in loader_all:
            le = batch['left_eye'].to(device)
            re = batch['right_eye'].to(device)
            f  = batch['face'].to(device)
            lb = batch['label'].to(device)

            if model_name in MODELS_WITH_GEO:
                geo = batch['geo_features'].to(device)
                out = m(le, re, f, geo)
            else:
                out = m(le, re, f)

            _, pred = torch.max(out, 1)
            preds.extend(pred.cpu().numpy())
            labels.extend(lb.cpu().numpy())

    acc = 100 * np.mean(np.array(preds) == np.array(labels))
    results_summary.append({'model': model_name, 'accuracy': acc})
    print(f"  {model_name}: {acc:.2f}%")

# Summary table
print(f"\n{'='*50}")
print("CEAL Cross-Dataset Evaluation Summary")
print(f"{'='*50}")
print(f"{'Model':<12} {'Accuracy':>10} {'vs iTracker':>12}")
print(f"{'-'*12} {'-'*10} {'-'*12}")
print(f"{'iTracker':<12} {'39.00%':>10} {'baseline':>12}")
for r in results_summary:
    delta = r['accuracy'] - 39.0
    print(f"{r['model']:<12} {r['accuracy']:>9.2f}% {delta:>+11.2f}%")

In [ ]:
# ============================================================
# SAVE PREDICTIONS for downstream analysis
# ============================================================

output_path = f"/Volumes/Crucial X10/210/ceal_predictions_{ACTIVE_MODEL}.csv"
df_results.to_csv(output_path, index=False)
print(f"Saved predictions to: {output_path}")
print(f"Columns: {list(df_results.columns)}")